# CELDA 1 — Carga y verificación del pipeline guardado

Cargamos el pipeline guardado y verificamos que está listo para producción.

In [ ]:
# =============================================================================
# CELDA 1 — Carga y verificación del pipeline guardado
# =============================================================================

# Importaciones principales
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import warnings

# Configuración de visualización
plt.style.use('seaborn-v0_8-darkgrid')
sns.set(font_scale=1.2)
warnings.filterwarnings('ignore')

# Añadir directorio src al path
sys.path.append(os.path.join('..', 'src'))

# Cargar pipeline
ruta_modelo = os.path.join('..', 'models', 'modelo_final.pkl')
pipeline = joblib.load(ruta_modelo)

print("Pipeline cargado exitosamente.")
print("\nComponentes del pipeline:")
for nombre, paso in pipeline.named_steps.items():
    print(f"  {nombre}: {type(paso).__name__}")

# Verificar que el modelo está listo para producción
print("\nVerificación del pipeline:")
print(f"  Número de pasos: {len(pipeline.steps)}")
print(f"  Modelo final: {type(pipeline.named_steps['modelo']).__name__}")
print("\n✓ Pipeline listo para producción.")

# CELDA 2 — Prueba del pipeline con datos nuevos de ejemplo

Construimos manualmente vectores de features de ejemplo (uno por sector)
y ejecutamos el pipeline para verificar que funciona correctamente.

In [ ]:
# =============================================================================
# CELDA 2 — Prueba del pipeline con datos nuevos de ejemplo
# =============================================================================

# Cargar dataset para obtener nombres de features
ruta_datos = os.path.join('..', 'data', 'processed', 'dataset_modelamiento.csv')
df = pd.read_csv(ruta_datos, index_col=0)

# Obtener nombres de features (excluyendo targets y sectores)
columnas_excluir = [col for col in df.columns if col.startswith('target_')] + \
                   [col for col in df.columns if col.endswith('_sector')]
nombres_features = [col for col in df.columns if col not in columnas_excluir]

print(f"Número de features: {len(nombres_features)}")
print(f"Features: {nombres_features}")

# Construir vectores de ejemplo por sector
ejemplos = {
    'energía': {
        'BRENT': 0.025,
        'WTI': 0.020,
        'EXXON': 0.015,
        'CHEVRON': 0.018,
        'VIX': -0.010,
        'SP500': 0.005,
        'BOVESPA': 0.003,
        'MERVAL': 0.002,
        'USD_COP': -0.005,
        'GOLD': 0.010,
        'COPPER': 0.008
    },
    'índice': {
        'SP500': 0.010,
        'BOVESPA': 0.015,
        'MERVAL': 0.012,
        'VIX': 0.005,
        'BRENT': 0.008,
        'WTI': 0.007,
        'USD_COP': -0.003,
        'GOLD': 0.005,
        'COPPER': 0.004,
        'EXXON': 0.006,
        'CHEVRON': 0.006
    },
    'divisa': {
        'USD_COP': 0.030,
        'VIX': 0.015,
        'BRENT': 0.010,
        'WTI': 0.009,
        'SP500': 0.002,
        'BOVESPA': 0.001,
        'MERVAL': 0.001,
        'GOLD': 0.003,
        'COPPER': 0.002,
        'EXXON': 0.004,
        'CHEVRON': 0.004
    },
    'metal': {
        'GOLD': 0.018,
        'COPPER': 0.012,
        'VIX': 0.008,
        'BRENT': 0.006,
        'WTI': 0.005,
        'SP500': 0.003,
        'BOVESPA': 0.002,
        'MERVAL': 0.002,
        'USD_COP': -0.002,
        'EXXON': 0.004,
        'CHEVRON': 0.004
    },
    'volatilidad': {
        'VIX': 0.045,
        'BRENT': -0.010,
        'WTI': -0.012,
        'SP500': -0.015,
        'BOVESPA': -0.020,
        'MERVAL': -0.018,
        'USD_COP': 0.025,
        'GOLD': 0.020,
        'COPPER': -0.008,
        'EXXON': -0.009,
        'CHEVRON': -0.009
    }
}

# Crear DataFrame con ejemplos
df_ejemplos = pd.DataFrame(index=ejemplos.keys(), columns=nombres_features)

# Llenar con valores por defecto (mediana del dataset)
for col in nombres_features:
    df_ejemplos[col] = df[col].median()

# Actualizar con valores específicos de cada ejemplo
for sector, valores in ejemplos.items():
    for feature, valor in valores.items():
        if feature in df_ejemplos.columns:
            df_ejemplos.loc[sector, feature] = valor

# Cargar scaler del entrenamiento
ruta_scaler = os.path.join('..', 'models', 'scaler.pkl')
scaler = joblib.load(ruta_scaler)

# Escalar los datos de ejemplo
df_ejemplos_std = pd.DataFrame(
    scaler.transform(df_ejemplos),
    index=df_ejemplos.index,
    columns=df_ejemplos.columns
)

# Ejecutar pipeline
print("\nEjecutando pipeline con datos de ejemplo escalados...")
predicciones = pipeline.predict(df_ejemplos_std)
probabilidades = pipeline.predict_proba(df_ejemplos_std)

# Mostrar resultados
print("\nResultados de predicción:")
print("="*80)

resultados = []
for i, sector in enumerate(ejemplos.keys()):
    pred = predicciones[i]
    prob_subida = probabilidades[i, 1]
    prob_bajada = probabilidades[i, 0]
    
    resultados.append({
        'Sector': sector,
        'Predicción': 'Subida' if pred == 1 else 'Bajada',
        'P(Subida)': f"{prob_subida:.2%}",
        'P(Bajada)': f"{prob_bajada:.2%}"
    })
    
    print(f"\n{sector.upper()}:")
    print(f"  Predicción: {'SUBIDA' if pred == 1 else 'BAJADA'}")
    print(f"  P(Subida): {prob_subida:.2%}")
    print(f"  P(Bajada): {prob_bajada:.2%}")

# Crear tabla de resultados
df_resultados = pd.DataFrame(resultados)
print("\nTabla de resultados:")
print(df_resultados.to_string(index=False))

## ¿Por qué usamos el índice de Youden?

El índice de Youden (también conocido como J Statistic) es una métrica
que busca el equilibrio óptimo entre la sensibilidad (TPR) y la especificidad
(1 - FPR). Se define como:

**Youden = TPR - FPR**

Un threshold óptimo maximiza esta diferencia, lo que resulta en:
- Una buena capacidad para detectar positivos (Recall/TPR alto)
- Una buena capacidad para evitar falsos positivos (Precision alta)

Al usar el threshold óptimo en lugar del defecto (0.50), logramos
balancear mejor la Precision y el Recall, evitando que uno esté muy
alto y el otro muy bajo (como ocurrió con Recall=0.99 y Precision baja).

**Objetivo:** Obtener Precision y Recall ambos entre 0.55 y 0.75.

In [ ]:
# =============================================================================
# CELDA 3 — Métricas finales del modelo en producción
# =============================================================================

# Importar módulo de evaluación y forzar recarga
import importlib
import evaluation as ev
importlib.reload(ev)
print("Módulo evaluation recargado")

import joblib
import os
from sklearn.metrics import roc_curve

# Cargar datos de prueba
ruta_datos = os.path.join('..', 'data', 'processed', 'dataset_modelamiento.csv')
df = pd.read_csv(ruta_datos, index_col=0)

# Preparar datos
activo_objetivo = 'BRENT'
target_col = f'target_{activo_objetivo}'

# Excluir columnas de target y sectores
columnas_excluir = [col for col in df.columns if col.startswith('target_')] + \
                   [col for col in df.columns if col.endswith('_sector')]
X = df.drop(columnas_excluir, axis=1)
y = df[target_col]

# Dividir datos (usar misma semilla que en entrenamiento)
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Cargar scaler del entrenamiento y aplicarlo a X_test
ruta_scaler = os.path.join('..', 'models', 'scaler.pkl')
scaler = joblib.load(ruta_scaler)
X_test_std = pd.DataFrame(scaler.transform(X_test), index=X_test.index, columns=X_test.columns)

# Recuperar pipeline guardado
ruta_modelo = os.path.join('..', 'models', 'modelo_final.pkl')
pipeline = joblib.load(ruta_modelo)

# -------------------------------------------------------------------------
# CALCULAR THRESHOLD ÓPTIMO USANDO ÍNDICE DE YOUDEN
# -------------------------------------------------------------------------
y_proba = pipeline.predict_proba(X_test_std)[:, 1]

# Calcular curva ROC
fpr, tpr, thresholds = roc_curve(y_test, y_proba)

# Calcular Índice de Youden
youden_index = tpr - fpr

# Encontrar threshold óptimo
idx_optimo = youden_index.argmax()
threshold_optimo = thresholds[idx_optimo]

print(f"\nThreshold óptimo (Índice de Youden): {threshold_optimo:.4f}")
print(f"  TPR en óptimo: {tpr[idx_optimo]:.3f}")
print(f"  FPR en óptimo: {fpr[idx_optimo]:.3f}")
print(f"  Youden máximo: {youden_index[idx_optimo]:.3f}")

# Hacer predicciones manuales
y_pred_optimo = (y_proba >= threshold_optimo).astype(int)

print(f"\nPredicciones con threshold óptimo ({threshold_optimo:.4f}):")
print(f"  Subidas predichas: {y_pred_optimo.sum()}")
print(f"  Bajadas predichas: {(1-y_pred_optimo).sum()}")

# -------------------------------------------------------------------------
# CALCULAR MÉTRICAS USANDO PREDICCIONES MANUALES
# -------------------------------------------------------------------------
# Pasar y_pred_optimo a ev.calcular_metricas_completas()
metricas = ev.calcular_metricas_completas(
    pipeline, X_test_std, y_test, 'Modelo Final', y_pred_optimo
)

# Comparar contra línea base
linea_base = 0.50  # Línea base de clasificación aleatoria (0.50)

print("\n" + "="*80)
print("COMPARACIÓN CONTRA LÍNEA BASE")
print("="*80)

print(f"\nLínea base de referencia: {linea_base:.2f}")
print(f"\nMétricas del modelo:")
print(f"  AUC-ROC: {metricas['auc']:.4f} (diferencia: {metricas['auc'] - linea_base:+.4f})")
print(f"  F1-Score: {metricas['f1']:.4f}")
print(f"  Accuracy: {metricas['accuracy']:.4f}")
print(f"  Precision: {metricas['precision']:.4f}")
print(f"  Recall: {metricas['recall']:.4f}")

# Conclusión
print("\nConclusión:")
if metricas['auc'] > linea_base:
    print(f"✓ El modelo supera la línea base en {metricas['auc'] - linea_base:.4f} puntos.")
    print("  El modelo agrega valor predictivo significativo.")
else:
    print(f"⚠ El modelo no supera la línea base.")
    print("  Se recomienda revisar el modelo o las features.")

In [ ]:
# =============================================================================
# CELDA 4 — Lanzamiento de la app Streamlit desde el notebook
# =============================================================================

import subprocess
import time

# Verificar si Streamlit está instalado
try:
    import streamlit
    print("Streamlit ya está instalado.")
except ImportError:
    print("Instalando Streamlit...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "streamlit"])

# Lanzar app Streamlit
print("\nLanzando aplicación Streamlit...")
print("La app se abrirá en el navegador en: http://localhost:8501")
print("\nPara detener la app, presione Ctrl+C en la terminal.")

# Cambiar al directorio de la app
os.chdir(os.path.join('..', 'app'))

# Lanzar Streamlit en segundo plano
proceso = subprocess.Popen(
    ['streamlit', 'run', 'streamlit_app.py'],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

print(f"\nProceso Streamlit iniciado con PID: {proceso.pid}")
print("Esperando 5 segundos para que la app se inicie...")
time.sleep(5)

# Verificar si el proceso está corriendo
if proceso.poll() is None:
    print("✓ App Streamlit corriendo correctamente.")
    print("\nPara acceder a la app, abra su navegador en: http://localhost:8501")
else:
    print("⚠ Error al iniciar la app Streamlit.")
    print("\nSalida de error:")
    print(proceso.stderr.read().decode())

# Documentación para reproducir el despliegue
print("\n" + "="*80)
print("PASOS PARA REPRODUCIR EL DESPLIEGUE DESDE CERO")
print("="*80)

print("\n1. Clonar el repositorio:")
print("   git clone <url_del_repositorio>")
print("   cd proyecto_maduro_mercados")"
